# image segmentation in python
a notebook to learn more advanced image segmentation

In [ ]:
pip install readlif

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

In [ ]:
from matplotlib import pyplot as plt
from readlif.reader import LifFile
from skimage import filters
from skimage.filters import try_all_threshold, gaussian
import numpy as np

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

as before we will load the same gephyrin image

In [ ]:
lif = LifFile('../data/2026group2/Gruppe2 WT - DMet High Density.lif')
image = lif.get_image(0)
channels_array = np.stack([np.array(channel) for channel in image.get_iter_c(t=0, z=0)])

gphn = channels_array[3]

plt.imshow(gphn, cmap='gray')


to separate the image into foreground and background, we need to define a cutoff intensity value

In [ ]:
threshold = 1000

binary_image = gphn >= threshold

plt.imshow(binary_image, cmap='gray')

there are many algorithms that can be used to calculate the threshold value, e.g. "Otsu" is one method

In [ ]:
threshold = filters.threshold_otsu(gphn)

threshold

as before we can use this computed value instead of an arbitrary chosen one to binarize the image

In [ ]:
binary_image = gphn >= threshold

plt.imshow(binary_image, cmap='gray')

in order to test how well this worked, we can plot the contour of the binary image onto the original

In [ ]:
# create a new plot
fig, axes = plt.subplots(1,1)

# add two images
axes.imshow(gphn, cmap=plt.cm.gray)
axes.contour(binary_image, [0.5], linewidths=1.2, colors='r')

we can test multiple thresholding methods to see how they display diffrent properties of the original image

In [ ]:
fig, ax = try_all_threshold(gphn, figsize=(10, 8), verbose=False)
plt.show()

finally we can test how image preprocessing can affect thresholding

In [ ]:
def sub_to_zero(a, b):
    # Element-wise subtraction and maximum with zero
    return np.maximum(a - b, 0)


# Process Gphn channel (Difference of Gaussian)
low = gaussian(gphn, sigma=2, preserve_range=True)
high = gaussian(gphn, sigma=10, preserve_range=True)

dog = sub_to_zero(low,high)

fig, ax = try_all_threshold(dog, figsize=(10, 8), verbose=False)
plt.show()

let's pick Otsu compare the effect of preprocessing

In [ ]:
threshold = filters.threshold_otsu(gphn)
binary_image = gphn >= threshold

threshold_dog = filters.threshold_otsu(dog)
binary_image_dog = dog >= threshold_dog



fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(gphn, cmap=plt.cm.gray)
axes[0].contour(binary_image, [0.5], linewidths=1.2, colors='r')
axes[0].set_title("threshold on original")

axes[1].imshow(gphn, cmap=plt.cm.gray)
axes[1].contour(binary_image_dog, [0.5], linewidths=1.2, colors='r')
axes[1].set_title("threshold on DoG")

plt.tight_layout()
plt.show()


... you can see how from here multiple rounds of trial and error could lead to success. Let's dicuss an entirely diffrent approach....